# 1. Comparação de modelos para risco de aumento
O objetivo é estimar a probabilidade de alta superior a 2% no mês-calendário seguinte. Quatro famílias de classificadores disputam o problema sob o mesmo pré-processamento e o mesmo critério de seleção: regressão logística, árvore de decisão, random forest e gradient boosting por histograma.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from basket_price_prediction.config import ROOT, DATASET, SPLITS, FEATURES, MIN_PRECISION, THRESHOLDS
from basket_price_prediction.data import load_data
from basket_price_prediction.preprocessing import prepare_monthly
from basket_price_prediction.features import build_supervised, temporal_split
from basket_price_prediction.training import GRID
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 2. Alvo e atributos
Para cada série, o alvo é 1 somente se `(preço t+1 / preço t) - 1 > 0,02`; exatamente 2% é 0. O modelo vê preços em t, t−1 e t−2, variações, média, desvio padrão populacional, mínimo e máximo dessa janela, mês cíclico e identificadores. As três observações devem ser consecutivas. O preço futuro e a variação efetiva são apenas avaliação.

In [ ]:
raw=load_data(); monthly,cleaning=prepare_monthly(raw); supervised,report=build_supervised(monthly)
display(pd.Series(report,name="n").to_frame()); display(supervised.head())

### 2.1 Por que o limite é 2%
O corte não é arbitrário. A distribuição das variações mensais mostra uma faixa central de reprecificação rotineira: a mediana é exatamente zero e perto de 44% das transições ficam dentro de ±2%. O limite também é economicamente relevante, já que 2% ao mês compostos equivalem a cerca de 27% ao ano.

In [ ]:
change=supervised.actual_change
faixas=pd.Series({"queda > 2%":(change<-.02).mean(),"estável em ±2%":(change.abs()<=.02).mean(),"alta > 2%":(change>.02).mean()},name="participação")
display(faixas.to_frame().style.format("{:.1%}"))
print("mediana da variação mensal: %.4f" % change.median())
print("2%% ao mês compostos em 12 meses: %.1f%%" % (100*(1.02**12-1)))
plt.figure(figsize=(9,3.5)); sns.histplot(change.clip(-.4,.4),bins=80)
for x in (-.02,.02): plt.axvline(x,color="crimson",ls="--")
plt.xlabel("Variação do preço no mês seguinte"); plt.ylabel("Frequência"); plt.title("Faixa de ±2% destacada"); plt.show()

from basket_price_prediction.evaluation import risk_by_variation_band
perfil=risk_by_variation_band(supervised)
display(perfil.style.format({"risk":"{:.3f}"}).hide(axis="index"))
display(Markdown(f"Preço inalterado concentra **{perfil.set_index('band').loc['preço inalterado','n']:,}** observações e tem o **menor** risco da tabela: preço de supermercado é rígido. O risco máximo está na faixa de queda acima de 20%, sinal de reversão à média."))

## 3. Partição cronológica e desequilíbrio
Os limites se referem ao **mês-alvo**; as características vêm até o mês anterior. Treino termina em dezembro/2024, validação cobre janeiro–agosto/2025 e teste setembro/2025–agosto/2026.

In [ ]:
parts=temporal_split(supervised,SPLITS)
display(pd.DataFrame({k:{"n":len(v),"positivos":int(v.target.sum()),"prevalência":v.target.mean(),"primeiro alvo":v.target_month.min(),"último alvo":v.target_month.max()} for k,v in parts.items()}).T)

## 4. As famílias comparadas e o critério
`ColumnTransformer` ajusta codificação one-hot somente no treino, aceitando categorias novas; a escala numérica é aplicada apenas à família linear, porque árvores são invariantes a transformação monotônica dos atributos.

O critério de seleção é único e vale para todas as famílias: **maior F1 na validação entre os candidatos com precisão de pelo menos 0,35**, com desempate por precisão, menor limiar e nome do algoritmo. Se nenhum candidato atingir o piso, usa-se o maior F1 e registra-se a exceção. O mesmo critério é reaplicado dentro de cada família para montar a comparação, de modo que nenhuma delas seja avaliada por uma régua mais favorável.

In [ ]:
grade=pd.DataFrame([{"família":k,"configurações":len(v),"exemplo de configuração":v[0]} for k,v in GRID.items()])
display(grade)
print("Total: %d configurações x %d limiares = %d candidatos avaliados na validação" % (grade.configurações.sum(),len(THRESHOLDS),grade.configurações.sum()*len(THRESHOLDS)))
print("Piso de precisão exigido na seleção: %.2f" % MIN_PRECISION)

## 5. Treinamento intencional e artefatos
Para gerar uma execução nova, defina `RUN_TRAINING = True` ou execute o notebook com `BASKET_TRAIN=1`. Cada execução cria um diretório único; por padrão a reabertura não treina. A configuração escolhida é reajustada em treino + validação e só então o teste é consultado. Predições de validação são do modelo anterior ao reajuste. Uma execução completa leva cerca de 100 segundos.

In [ ]:
import os
from basket_price_prediction.experiments import run_training, load_experiment
RUN_TRAINING = os.environ.get("BASKET_TRAIN", "0") == "1"
if RUN_TRAINING:
    experiment_path=run_training()
    print("Experimento concluído:",experiment_path)
else:
    print("Treinamento não solicitado. Defina RUN_TRAINING=True para criar uma execução.")

## 6. Evidências da seleção
A tabela de candidatos vem exclusivamente da validação. A comparação entre famílias mostra o melhor candidato de cada uma sob o mesmo critério: é essa coluna que decide a escolha, e não qualquer métrica de teste.

In [ ]:
if RUN_TRAINING:
    experiment=load_experiment(experiment_path)
    display(pd.Series(experiment["config"]["selection"],name="seleção").to_frame())
    comparacao=experiment["comparison"]
    display(comparacao[["algorithm","params","threshold","precision_validation","recall_validation","f1_validation"]].sort_values("f1_validation",ascending=False))
    display(pd.read_csv(experiment_path/"reports"/"validation_candidates.csv").sort_values("f1",ascending=False).head(10))
    display(pd.Series(experiment["metrics"]["validation_before_refit"],name="validação antes do reajuste").to_frame())

A família vencedora é decidida por evidência, e não por hipótese inicial. O ajuste bem-sucedido, por si só, não demonstra utilidade preditiva: consulte o teste independente, os intervalos de confiança, os falsos alarmes, a estabilidade temporal e as limitações no notebook seguinte.